# Ask Hestia · Assignment 8 RAG experiments
Five household questions, three chunking strategies and three embedding encoders. Each of the 45 requests uses the same Qwen2 generator. The tables below read the measured runs and the fact-by-fact answer review. Set the inference switch to repeat the experiment on your hardware.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
root = Path.cwd()
if (root / "corpus.py").is_file() and root.name == "a8":
    root = root.parent
sys.path.insert(0, str(root))
from a8.corpus import load_chunks, load_queries, STRATEGIES
from a8.person3_experiments import OUT, run
from a8.person3_analysis import evaluation_table, summary, SIZES
pd.DataFrame([{"strategy": s, "full chunks": len(load_chunks(s, "full")),
               "subset chunks": len(load_chunks(s, "small"))} for s in STRATEGIES])

,strategy,full chunks,subset chunks
0,fixed,30,15
1,overlap,20,10
2,hybrid,20,10


### Generation and retrieval
The query is embedded and matched to indexed handbook chunks by cosine similarity. The generator receives its system instruction, the unchanged top three passages in rank order, and the original question. The answer key is used only after decoding. Model loading, index construction, one warm-up per configuration and scoring are outside the measured latency.

In [2]:
from a8.generation import MODEL, REVISION, SYSTEM, MAX_NEW_TOKENS
print(MODEL, REVISION)
print(SYSTEM)
print(f"Greedy decoding, maximum {MAX_NEW_TOKENS} new tokens.")
display(pd.DataFrame(load_queries())[["id", "query"]])

Qwen/Qwen2-0.5B-Instruct c540970f9e29518b1d8f06ab8b24cba66ad77b6d
You answer questions about the Hestia household handbook. Use only the cited passages. Answer all parts of the question directly, preserving the names, quantities and conditions in the handbook. If a detail is missing, say that the handbook does not specify it.
Greedy decoding, maximum 128 new tokens.


,id,query
0,Q1,"I finished closing the kitchen, but the dishwa..."
1,Q2,The store ran out of an item on our shared gro...
2,Q3,I need the shared cordless drill this afternoo...
3,Q4,The bathroom tap has a slow drip even with the...
4,Q5,A friend is staying overnight. What needs to b...


In [3]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    # Fresh raw results are kept separate from the reviewed reference experiment.
    for size in SIZES:
        run(size, OUT.parent / "repeat" / size)

### Evaluation table
Context quality is the percentage of required evidence quotes found in the retrieved texts. Answer quality is the percentage of fully conveyed required facts after deductions for unsupported instructions. See person3/RUBRIC.md and results/person3/reviews.csv for grading decisions. End-to-end timing measures an actual request.

In [4]:
rows = evaluation_table()
table = pd.DataFrame(rows)
display(table[["size", "strategy", "query_id", "retrieved_context_quality", "answer_quality",
               "end_to_end_ms", "notes"]].round(2).style.hide(axis="index"))

size,strategy,query_id,retrieved_context_quality,answer_quality,end_to_end_ms,notes
small,fixed,Q1,100.000000,66.670000,4729.640000,Completes the card and defers unloading to the morning resident. Assumes the tag is already attached.
small,fixed,Q2,100.000000,0.000000,1638.910000,Omits both required actions and suggests closing the grocery entry.
small,fixed,Q3,100.000000,0.000000,6005.480000,"Never identifies the blue bin or gives the complete checkout record. Directs the reader to look up the location on the card, log a check time, and log location/status fields."
small,fixed,Q4,100.000000,0.000000,7032.720000,Identifies the Plumbing card but never says to reuse it or records the three details. Makes the photo mandatory and conditions notification on the card being unavailable. Output ends mid-sentence.
small,fixed,Q5,100.000000,20.000000,3997.310000,"Gives a sheet set. Leaves out exact towel and carafe placement, the lamp check and the completion rule."
small,overlap,Q1,100.000000,100.000000,6361.770000,"All three actions appear, although the answer describes them in the first person."
small,overlap,Q2,100.000000,0.000000,1563.710000,Quantity alone does not say to leave the item unchecked or write Out of stock.
small,overlap,Q3,100.000000,0.000000,5017.360000,"Omits the borrower name, bin and closure. Sends the reader to the checkout card for the location and to appliance upkeep for the return."
small,overlap,Q4,100.000000,75.000000,6174.520000,"Reuses the existing card, records all three details and notifies through it. Does not name the Plumbing card on the Maintenance board."
small,overlap,Q5,100.000000,100.000000,7384.420000,"All required quantities, placements, lamp check and completion condition appear before the final sentence cuts off."


### Inspect an individual request
Change the row ID to inspect the exact prompt, full ranked chunks, generated answer and reference for any of the 45 requests.

In [5]:
row_id = "medium/full/hybrid/Q3"
size = row_id.split("/")[0]
records = [json.loads(line) for line in (OUT / size / "rows.jsonl").read_text().splitlines()]
example = next(r for r in records if r["row_id"] == row_id)
print(example["prompt"])
print("\nGenerated answer:", example["answer"])
print("\nReference:", next(q["expected_answer"] for q in load_queries() if q["id"] == example["query_id"]))

<|im_start|>system
You answer questions about the Hestia household handbook. Use only the cited passages. Answer all parts of the question directly, preserving the names, quantities and conditions in the handbook. If a detail is missing, say that the handbook does not specify it.<|im_end|>
<|im_start|>user
Handbook passages:
[hybrid-inventory-01]
The shared tool inventory identifies equipment that belongs to the household and its normal storage location. Small hand tools are arranged in labeled bins so a resident can tell whether a missing item is being used or was put away elsewhere. The inventory card includes the tool name, its home bin, and any accessories that travel with it. Before starting a project, check the card and confirm that the necessary pieces are present. Residents leave personal equipment in their own storage unless they have agreed to share it. This keeps household availability records meaningful for everyone.

Before borrowing the shared cordless drill, add your nam

### Embedding comparison

In [6]:
display(pd.DataFrame(summary(rows)).round(2))
display(Markdown((root / "a8" / "person3" / "COMPARISON.md").read_text()))

,size,dimension,runs,mean_context,mean_answer,rank_one,median_retrieval_ms,median_request_ms
0,small,384,15,100.0,37.44,15,9.10,4729.64
1,medium,768,15,100.0,33.33,15,27.82,4114.95
2,large,1024,15,100.0,41.89,15,84.48,3492.91


# Step 2.3 · Does a larger embedding improve Hestia's answers?

These are matched runs of five handbook questions under fixed, overlap and hybrid chunking. Only the embedding encoder and its specified query formatting vary within a matched query/strategy comparison. The full corpus, top three, generator, prompt format, greedy decoding, output limit and CPU thread count are fixed. Each mean gives each of the 15 rows equal weight. Scores are on a 0–100 scale under the shared rubric.

| Embedding model | Dimensions | Runs | Evidence coverage | First supporting hit at rank 1 | Mean answer quality | Median retrieval | Median full request |
|---|---:|---:|---:|---:|---:|---:|---:|
| all-MiniLM-L6-v2 | 384 | 15 | 100% | 15/15 | 37.44 | 9.1 ms | 4.73 s |
| all-mpnet-base-v2 | 768 | 15 | 100% | 15/15 | 33.33 | 27.8 ms | 4.11 s |
| bge-large-en-v1.5 | 1,024 | 15 | 100% | 15/15 | 41.89 | 84.5 ms | 3.49 s |

**Retrieval quality.** For every encoder, the first hit referenced a supporting paragraph ID, and the three passages together contained every required quote. A fixed-length chunk can contain only part of a paragraph. Inspecting the first hit alone, it contained all required quotes on 11/15 small, 10/15 medium and 10/15 large requests. On the fixed grocery question, the first hit contained none of the required quote for all three encoders. With these five questions there was no measurable gain in those two retrieval metrics from increasing dimension. The identities of later passages could still change. Those passages are part of the generator's input.

**Answer quality.** BGE had the highest mean, while MPNet scored below MiniLM. On the out-of-stock question with overlapping chunks, the small encoder's answer just said to record the quantity. The medium and large runs both said to leave the original item unchecked and write “Out of stock,” earning 100. With fixed chunks on the same question, all three scored zero despite retrieving the rule. The prompt and raw passages for each case are in the JSONL files.

**Did larger always win?** No. On the kitchen handoff with fixed chunks, the small run scored 66.67, the medium 0 and the large 33.33. For preparing a guest room with hybrid chunks, all three covered every required fact. Embedding dimension alone cannot explain these observations: the encoders differ in training, and the generator can overlook or misuse evidence. The 128-token cap was reached on 11 of 45 requests; the reviews score only what the model actually returned.

**Latency.** Median retrieval latency grew with encoder size on this CPU. End-to-end medians did not follow the same order. The full request includes variable-length generation. There is one timing per configuration/query after warm-up, models were run in sequence, and five questions from one handbook are insufficient to establish a general speed or quality ranking.

The per-answer scores are an AI-assisted review of actual generated text. The matched facts, deductions, review notes and evidence fingerprints are in results/person3/reviews.csv. The full evaluation table includes the retrieved texts and actual end-to-end latency for each request.
